# Decoding

In [1]:
from collections import defaultdict

import muspy
import torch
import torch.nn.functional as F
from midiutil import MIDIFile

from raag_midi_gen.tokenization.encoding import EventType
from raag_midi_gen.utils.midi_utils import play_muspy_music

In [2]:
DEFAULT_TEMPO = 120
DEFAULT_TPQ = 96

In [3]:
def _initialize_muspy_midi(tempo, tpq) -> muspy.Music:
    my_midi = muspy.Music()
    
    my_midi.resolution = tpq
    my_midi.tempos = [muspy.Tempo(0, tempo)]
    my_midi.time_signatures = [muspy.TimeSignature(0, 4, 4)]
    my_midi.tracks = [muspy.Track(program=0, is_drum=False, notes = [])]

    return my_midi


def decode_output(
    velocities: torch.Tensor,
    pitches: torch.Tensor,
    octaves: torch.Tensor,
    event_types: torch.Tensor,
    tempo: int = DEFAULT_TEMPO,
    tpq: int = DEFAULT_TPQ
):
    notes_tracker = {}
    final_midi = _initialize_muspy_midi(tempo, tpq)    

    len_in_ticks = velocities.shape[0]
    midi_velocities = (velocities.detach() * 127).clamp(0,127).round().long()
    
    for t in range(len_in_ticks):
        event_type = EventType(torch.argmax(event_types[t]).item())
    
        if not event_type in [EventType.NO_NOTE, EventType.NOTE_HOLD]:
            pitch  = torch.argmax(pitches[t][1:]).item()  # ignore the first index of the output pitch tensor, which corresponds to NO_NOTE
            octave = torch.argmax(octaves[t][1:]).item()  # ignore the first index of the output pitch tensor, which corresponds to NO_NOTE
            midi_pitch = octave*12 + pitch
    
            current_velocity = midi_velocities[t].item()
    
            if event_type == EventType.NOTE_ON:
                if midi_pitch in notes_tracker:  # Close if there is an existing note
                    on_timestep, velocity = notes_tracker[midi_pitch]
                    duration = t - on_timestep
                    final_midi.tracks[0].notes.append(muspy.Note(time=on_timestep, duration=duration, pitch=midi_pitch, velocity=velocity))
                notes_tracker[midi_pitch] = (t, current_velocity)
            if event_type == EventType.NOTE_OFF:
                if midi_pitch in notes_tracker:  # Add note to final_midi if there is a corresponding NOTE_ON event
                    on_timestep, velocity = notes_tracker[midi_pitch]
                    duration = t - on_timestep
                    final_midi.tracks[0].notes.append(muspy.Note(time=on_timestep, duration=duration, pitch=midi_pitch, velocity=velocity))
    
    # Finally, close out any unclosed notes. Default to a duration of a quarter note
    for midi_pitch, info in notes_tracker.items():
        on_timestep, velocity = info
        duration = tpq if on_timestep + tpq < len_in_ticks else len_in_ticks - on_timestep
        final_midi.tracks[0].notes.append(muspy.Note(time=on_timestep, duration=duration, pitch=midi_pitch, velocity=velocity))
    
    return final_midi

## Test It

In [4]:
import random

from raag_midi_gen.tokenization.encoding import encode_target
from raag_midi_gen.datasets.dataset import midi_files_dataset


midi_files_dataset = midi_files_dataset()
midi_filename, test_muspy_midi = midi_files_dataset[random.randint(1,100)]

target_encoding = encode_target(test_muspy_midi)

In [5]:
pitches = target_encoding['pitch']
octaves = target_encoding['octave']
velocities = target_encoding['velocity']
event_types = target_encoding['note_event_type']

In [6]:
decoded_midi = decode_output(velocities, pitches, octaves, event_types)

In [7]:
print(midi_filename)
print('------------------')
play_muspy_music(decoded_midi)

Taraana - Sthaayi 2.1.mid
------------------
FluidSynth runtime version 2.2.5
Copyright (C) 2000-2022 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file 'tmp.wav'..
